# Preprocessing Notebook - Google Colab

This notebook performs data preprocessing and visualization for the Pirate Pain Classification task on Google Colab.

**Setup:** The notebook will clone the repository and install dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/Gradient-Gang/ANN-Challenges.git
%cd ANN-Challenges

In [ ]:
# Install dependencies
!pip install -q numpy pandas scikit-learn matplotlib seaborn scipy torch torchvision lightning tensorboard optuna psycopg2-binary sqlalchemy ipywidgets optuna-dashboard litmodels python-dotenv

In [ ]:
# Add src to path for imports
import sys
sys.path.insert(0, '/content/ANN-Challenges/src')

In [ ]:
from GradientGang import *
from GradientGang.PreProcessing import PreProcessor
%load_ext autoreload
%autoreload 2

In [ ]:
params_path = "Notebook/Params/preprocessing_params.yaml"
params_path

In [ ]:
preprocessing = PreProcessor.fromYAML(params_path)
preprocessing.preprocess()

In [ ]:
trainData = preprocessing.load_data("dataset/PirateProcessed/pirate_pain_train.csv")
trainData.head()

In [ ]:
trainData["isPirate"].value_counts()

In [ ]:
trainLabels = preprocessing.load_data("dataset/PirateProcessed/pirate_pain_train_labels.csv")
trainLabels.head()

In [ ]:
# Check data shapes to understand the mismatch
print(f"Train Data shape: {trainData.shape}")
print(f"Train Labels shape: {trainLabels.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
sns.set_theme()

# Aggregate time series data by sample_index (take mean across time)
if 'sample_index' in trainData.columns:
    # Group by sample_index and take mean of all numeric columns
    trainDataAgg = trainData.groupby('sample_index').mean(numeric_only=True)
    # Remove time column if it exists
    if 'time' in trainDataAgg.columns:
        trainDataAgg = trainDataAgg.drop(columns=['time'])
    trainDataNp = trainDataAgg.to_numpy()
    print(f"Aggregated Train Data shape: {trainDataNp.shape}")
else:
    trainDataNp = trainData.to_numpy()

trainLabelsNp = trainLabels["label"].to_numpy()
print(f"Train Labels shape: {trainLabelsNp.shape}")

# Plot first 2 principal components colored by label
pca = PCA(n_components=2)
trainDataPca = pca.fit_transform(trainDataNp)
plt.figure(figsize=(10, 7))
sns.scatterplot(x=trainDataPca[:, 0], y=trainDataPca[:, 1], hue=trainLabelsNp, palette="Set1")
plt.title("PCA of Training Data Colored by Labels")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Label", loc="best")
plt.show()

In [ ]:
# Plot tsne colored by label
from sklearn.manifold import TSNE

# Use the aggregated data from previous cell
tsne = TSNE(n_components=2, random_state=42)
trainDataTsne = tsne.fit_transform(trainDataNp)
plt.figure(figsize=(10, 7))
sns.scatterplot(x=trainDataTsne[:, 0], y=trainDataTsne[:, 1], hue=trainLabelsNp, palette="Set1")
plt.title("t-SNE of Training Data Colored by Labels")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(title="Label", loc="best")
plt.show()